# Orquestrador Automático de Tutoria (PBL)
Este notebook extrai sumários de livros de medicina em PDF do Google Drive, mapeia objetivos usando a inteligência do Gemini (Antigravity SDK), e gera PDFs consolidados com capas premium.

In [ ]:
# 1. Instalação de Dependências
!pip install pymupdf pypdf reportlab pydantic python-dotenv google-antigravity "protobuf>=5.29.1,<6.0.0" --quiet
!pip install httpx==0.28.1 --quiet

In [ ]:
# 2. Montagem do Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuração da API do Gemini (Secrets)
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Chave configurada com sucesso!')

In [ ]:
# 4. Motor do Orquestrador
import asyncio
import logging
import json
import os
import io
import pydantic
import fitz
from google.antigravity import Agent, LocalAgentConfig
from pypdf import PdfReader, PdfWriter, PageObject
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import inch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

class Corte(pydantic.BaseModel):
    arquivo: str
    capitulo: str
    nivel: str
    pagina_inicial: int
    pagina_final: int

class Objetivo(pydantic.BaseModel):
    numero: str
    titulo: str
    cortes: list[Corte]

class RoteiroTutoria(pydantic.BaseModel):
    objetivos: list[Objetivo]

def draw_wrapped_text(c, text, width, x, y, font, size, line_height, color):
    c.setFont(font, size)
    c.setFillColor(color)
    words = text.split(' ')
    lines = []
    current_line = []
    for word in words:
        current_line.append(word)
        if c.stringWidth(' '.join(current_line), font, size) > width:
            current_line.pop()
            lines.append(' '.join(current_line))
            current_line = [word]
    if current_line:
        lines.append(' '.join(current_line))
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def create_cover_page(objetivo_numero, objetivo_titulo):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=A4)
    width, height = A4
    c.setFillColorRGB(0.1, 0.12, 0.15)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    c.setFillColorRGB(0.2, 0.6, 0.86)
    c.rect(0, 0, 20, height, fill=1, stroke=0)
    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 42)
    c.drawString(60, height - 120, f"OBJETIVO {objetivo_numero}")
    c.setFont("Helvetica", 18)
    c.setFillColorRGB(0.8, 0.8, 0.8)
    c.drawString(60, height - 160, "Roteiro de Tutoria - PBL")
    draw_wrapped_text(c, objetivo_titulo, width - 120, 60, height - 260, "Helvetica-Bold", 24, 32, colors.white)
    c.save()
    packet.seek(0)
    return PdfReader(packet).pages[0]

def create_separator_page(livro, capitulo):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=A4)
    width, height = A4
    c.setFillColorRGB(0.95, 0.96, 0.98)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    c.setStrokeColorRGB(0.2, 0.6, 0.86)
    c.setLineWidth(4)
    c.line(50, height/2 + 50, width-50, height/2 + 50)
    draw_wrapped_text(c, f"Livro: {livro}", width - 100, 50, height/2 + 80, "Helvetica", 14, 20, colors.darkgray)
    draw_wrapped_text(c, capitulo, width - 100, 50, height/2 - 20, "Helvetica-Bold", 20, 28, colors.black)
    c.save()
    packet.seek(0)
    return PdfReader(packet).pages[0]

def get_pdfs_tocs(folder_path):
    tocs = {}
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            filepath = os.path.join(folder_path, filename)
            try:
                doc = fitz.open(filepath)
                tocs[filename] = doc.get_toc()
                doc.close()
            except Exception as e:
                logging.error(f"Erro ao ler TOC de {filename}: {e}")
    return tocs

def merge_and_sort_cortes(cortes):
    if not cortes: return []
    from collections import defaultdict
    grouped = defaultdict(list)
    for c in cortes: grouped[c.arquivo].append(c)
    merged = []
    for arquivo, lista in grouped.items():
        lista.sort(key=lambda x: x.pagina_inicial)
        curr = lista[0]
        for nxt in lista[1:]:
            if nxt.pagina_inicial <= curr.pagina_final + 3:
                curr.pagina_final = max(curr.pagina_final, nxt.pagina_final)
            else:
                merged.append(curr)
                curr = nxt
        merged.append(curr)
    merged.sort(key=lambda x: {"conceito": 0, "mecanismo": 1, "clinica": 2}.get(x.nivel, 99))
    return merged

def gerar_pdfs(roteiro, pdfs_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for obj in roteiro.objetivos:
        obj_pdf_path = os.path.join(output_dir, f"Objetivo {obj.numero}.pdf")
        writer = PdfWriter()
        writer.add_page(create_cover_page(obj.numero, obj.titulo))
        cortes = merge_and_sort_cortes(obj.cortes)
        last_book_chapter = ""
        for corte in cortes:
            source_pdf = os.path.join(pdfs_dir, corte.arquivo)
            if not os.path.exists(source_pdf): continue
            curr_bc = f"{corte.arquivo}_{corte.capitulo}"
            if curr_bc != last_book_chapter:
                writer.add_page(create_separator_page(corte.arquivo, corte.capitulo))
                last_book_chapter = curr_bc
            reader = PdfReader(source_pdf)
            p_ini, p_fim = max(0, corte.pagina_inicial - 1), min(len(reader.pages), corte.pagina_final)
            for p_num in range(p_ini, p_fim): writer.add_page(reader.pages[p_num])
        with open(obj_pdf_path, "wb") as f: writer.write(f)
        logging.info(f"Gerado: {obj_pdf_path}")

async def process_roteiro(objetivos_text, tocs):
    contexto = "SUMÁRIOS:\n"
    for arquivo, toc in tocs.items():
        if toc:
            contexto += f"Livro: {arquivo}\n"
            for item in toc:
                contexto += f"{'  '*(item[0]-1)}- {item[1]} (Página: {item[2]})\n"
    system_prompt = """Você mapeia Objetivos de Aprendizagem para os capítulos corretos. 
REGRAS:
1. Use APENAS páginas fornecidas nos sumários.
2. pagina_final é a página da próxima seção menos 1.
3. Nível didático: 'conceito', 'mecanismo' ou 'clinica'."""
    api_key = os.environ.get("GEMINI_API_KEY")
    config = LocalAgentConfig(response_schema=RoteiroTutoria, system_instructions=system_prompt, model="gemini-1.5-pro", api_key=api_key)
    async with Agent(config) as agent:
        resp = await agent.chat(f"{contexto}\n\nOBJETIVOS:\n{objetivos_text}")
        return await resp.structured_output()

async def run_medhelp(objetivos_text, refs_dir, out_dir):
    tocs = get_pdfs_tocs(refs_dir)
    data = await process_roteiro(objetivos_text, tocs)
    if data:
        gerar_pdfs(data, refs_dir, out_dir)
        print('✅ Processamento 100% Finalizado!')


In [ ]:
# 5. EXECUÇÃO DA TUTORIA
# =========================
PASTA_LIVROS = "/content/drive/MyDrive/Logística - Drive/Tutoria/Referências - MEDICINA"
PASTA_SAIDA = "/content/drive/MyDrive/Logística - Drive/Tutoria/saida/Tutoria_Teste"

OBJETIVOS = """
1. Descrever as drogas ilícitas mais comuns...
2. Confrontar os conceitos clínicos e neurobiológicos...
"""

import nest_asyncio
nest_asyncio.apply()
import asyncio

asyncio.run(run_medhelp(OBJETIVOS, PASTA_LIVROS, PASTA_SAIDA))
